# Which tiles made the model call each tumour type

Settings: Accelerator = GPU, Internet = on. Secrets: `HF_TOKEN` and a fresh `EBRAINS_TOKEN` (it expires, copy it again from the browser).

Input: upload the `features` folder from the pilot (the `.pt` files) as a Kaggle dataset and set its path below.

What it draws, one figure per slide:
- **A**: the slide with each tile coloured by how much it pushed the model towards the type (red) or away from it (blue)
- **B**: the same tiles in Virchow embedding space
- **C**: the real H&E tiles that contributed most
- **D**: inside those tiles, where Virchow looked (attention)

The slide is downloaded again for C and D (about 1-2.5 GB), used, then deleted.

**Nothing is computed twice.** The H&E tiles and attention of each slide are saved in `explain/cache` and `ood/cache`. Re-running a cell in the same session reuses them. To keep them for a later session: click *Save Version* (or download `explain.zip` and upload it as a Kaggle dataset), then *Add Input* that output in the new session. The Step 1 cell finds any earlier `cache` folder under `/kaggle/input` and copies it back.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/AsmaActiveLearning/Cancer_diagnosis.git 2>/dev/null || git -C Cancer_diagnosis pull -q
%cd /kaggle/working/Cancer_diagnosis
!pip install -q -r requirements-kaggle.txt umap-learn

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"))
os.environ["EBRAINS_TOKEN"] = secrets.get_secret("EBRAINS_TOKEN")

Step 1: panels A-B for every slide (no download, a few minutes). The summary says which slides the model got right.

In [ ]:
FEATURES = "/kaggle/input/<your-features-dataset>/features"  # change this
OUT = "/kaggle/working/explain"
OOD = "/kaggle/working/ood"

# Reuse H&E tiles and attention saved by an earlier session (added with "Add Input"), so no slide is downloaded twice.
import glob, shutil
for old in glob.glob("/kaggle/input/**/cache", recursive=True):
    dest = f"{OOD}/cache" if "/ood/" in old else f"{OUT}/cache"
    shutil.copytree(old, dest, dirs_exist_ok=True)
    print("reusing", old)

%cd /kaggle/working/Cancer_diagnosis
!python -m brainpath.explain {FEATURES} --out {OUT}

import json, pandas as pd
summary = pd.DataFrame(json.load(open(f"{OUT}/explain_summary.json")))
summary["right"] = summary.pred == summary.tumour_type
summary[["slide_id", "tumour_type", "pred", "right", "prob_target", "positive_share"]]

Step 2: pick a few slides (for example one right and one wrong per type) and draw all four panels. Each slide is downloaded, explained, then deleted. Slides already in the cache are not downloaded again.

In [ ]:
SLIDES = summary.groupby(["tumour_type", "right"]).slide_id.first().tolist()[:6]  # or list slide ids yourself
print(SLIDES)

!mkdir -p /kaggle/tmp
!python -m brainpath.explain {FEATURES} --out {OUT} --slides /kaggle/tmp --download --only {" ".join(SLIDES)}

In [ ]:
from IPython.display import Image, display
for s in SLIDES:
    display(Image(f"{OUT}/{os.path.splitext(s)[0]}_explain.png"))

Step 3: unseen tumour types, grouped by how easy they are to detect

Each rare or held-out type is hidden from training in turn. Types are grouped as **far-OOD** (easy to detect), **intermediate** and **near-OOD** (looks like a known type). One slide per type is downloaded and drawn with its tile novelty, the known type each tile is closest to, Virchow embedding space, the most novel tiles and a caption that interprets the figure. Add `--all` for every slide of each hidden type.

In [ ]:
!python -m brainpath.ood_explain {FEATURES} --out {OOD} --slides /kaggle/tmp --download

import glob
display(Image(f"{OOD}/ood_overview.png"))
for f in sorted(glob.glob(f"{OOD}/*OOD_*.png")) + sorted(glob.glob(f"{OOD}/intermediate_*.png")):
    print(os.path.basename(f))
    display(Image(f))

Download `/kaggle/working/explain` from the Output panel (or zip it below).

In [ ]:
!cd /kaggle/working && zip -qr explain.zip explain ood && ls -lh explain.zip